# AI Portfolio Assistant – Baseline Version

This notebook contains the initial implementation of an AI portfolio assistant based on Retrieval-Augmented Generation (RAG).

The goal of this version was to build and validate the complete RAG pipeline before introducing optimization techniques.

Main components:

- repository loading
- notebook parsing
- chunking
- embeddings
- retrieval
- answer generation
- evaluation

## Imports and Environment Setup
This section imports libraries used for embeddings, vector databases, OpenAI API calls, evaluation, and the Gradio interface.

In [1]:
# imports
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from chromadb import PersistentClient
from pydantic import BaseModel, Field
import json
import re
import os
import requests
from pypdf import PdfReader
import gradio as gr
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import math
from litellm import completion
from collections import Counter

## Configuration
Model names, database names, and collection settings are defined here.

In [2]:
MODEL = "gpt-4.1-mini"
openai = OpenAI()
DB_NAME = "vector_db"
collection_name = "docs"
embedding_model = "text-embedding-3-small"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

RETRIEVAL_K = 20

OpenAI API Key exists and begins sk-proj-


## Notebook Parsing Utilities
Functions for loading `.ipynb` files and converting notebook cells into text for RAG indexing.

In [3]:
def load_ipynb(file_path):
    texts = []

    with open(file_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "markdown":
            texts.append("MARKDOWN:\n" + "".join(cell.get("source", [])))
        elif cell.get("cell_type") == "code":
            texts.append("CODE:\n" + "".join(cell.get("source", [])))

    return "\n".join(texts)

## Project Metadata Extraction
Utilities for detecting project names and organizing repository structure.

In [4]:
def extract_project_name(path: str) -> str:
    if not path:
        return "UNKNOWN_PROJECT"

    path = path.replace("\\", "/")

    match = re.search(r"repos/([^/]+)/", path)

    if match:
        return match.group(1)

    return "UNKNOWN_PROJECT"

## Repository Loading
Loads source files from repositories and converts them into LangChain documents.

In [5]:
def load_files():
    documents = []

    repo_path = "repos"

    for root, _, files in os.walk(repo_path):
        if ".git" in root:
            continue

        print("Checking:", root)

        for file in files:
            print("Found:", file)

            if file.endswith((".md", ".ipynb")):
                file_path = os.path.join(root, file)

                try:
                    if file.endswith(".ipynb"):
                        content = load_ipynb(file_path)
                    else:
                        with open(file_path, "r", encoding="utf-8") as f:
                            content = f.read()

                    project_name = extract_project_name(file_path)

                    documents.append(
                        Document(
                            page_content=content,
                            metadata={
                                "source": file_path,
                                "project": project_name
                            }
                        )
                    )

                except Exception as e:
                    print("ERROR:", file_path, e)

    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = load_files()

Checking: repos
Checking: repos\Career-chat-rag
Checking: repos\Car_brand_classification_CNN
Found: Car_brand_classification_CNN.ipynb
Found: README.md
Checking: repos\Cracow_Real_Estate_Price_Prediction_2025
Found: distance.csv
Found: Flats_feature_engineering.ipynb
Found: flats_result_set1.csv
Found: Flats_scraping.ipynb
Found: flats_set1.csv
Found: Models_results.ipynb
Found: README.md
Found: requirements.txt
Checking: repos\LIDAR_point_cloud_classification_ANN
Found: 78989_1475655_M-34-64-D-d-2-3-2-4.rar
Found: Lidar_point_cloud_classification.ipynb
Found: README.md
Checking: repos\NLP-Classification_of_tweets_about_airlines_LSTM
Found: Airlines_Sentiments_Analysis_LSTM.ipynb
Found: README.md
Found: training_twitter_x_y_train.csv
Loaded 10 documents


In [7]:
total_chars = sum(len(doc.page_content) for doc in documents)
print("Total characters:", total_chars)

chars_list= []
for doc in documents:
    chars_list.append(len(doc.page_content))
print("Characters in files:",chars_list)


Total characters: 113798
Characters in files: [16646, 738, 43359, 7645, 18526, 1032, 11684, 1266, 12087, 815]


## Chunking Strategy
Documents are split into overlapping chunks before embedding generation.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 155 chunks
First chunk:

page_content='CODE:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping 
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPool2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
CODE:
# Defining training, validation and test sets
train_data_dir = 'Data/Train/' 
valid_data_dir = 'Data/Validation/'
test_data_dir = 'Data/Test/'
CODE:
#Defining the image size, batch size and number of output classes.
h, w = 256, 256 
batch_size = 8 
n_classes = 8
MARKDOWN:
# Convolution neural netwo

## Embedding Generation
Embeddings are created and stored in ChromaDB.

In [9]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [10]:
# Main RAG answering pipeline
create_embeddings(chunks)

Vectorstore created with 155 documents


In [11]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)

## RAG Prompt Design
Defines the main RAG system prompt and helper formatting functions.

In [12]:
rag_system_prompt = """
You answer questions about GitHub projects.

RULES:
- Use ONLY provided context
- Never invent projects
- Group information by project
- If project is missing in context → it does not exist
- All projects in context belong to Joanna Waligóra (her GitHub portfolio)

Each chunk contains:
- project name
- source file

Use chunks for:
- implementation details
- architecture
- code behavior

CONTEXT:
{context}
"""

In [13]:
def make_rag_messages(question, history, chunks):

    context = "\n\n".join(
        f"[PROJECT: {c.metadata.get('project','UNKNOWN')}]\n"
        f"{c.page_content}"
        for c in chunks
    )

    system_prompt = rag_system_prompt.format(
        context=context
    )

    return [
        {"role": "system", "content": system_prompt},
        *history,
        {"role": "user", "content": question}
    ]

In [14]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [15]:
def fetch_context(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [16]:
def answer_question(question: str, history: list[dict] = None):
    if history is None:
        history = []

    chunks = fetch_context(question)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)

    return response.choices[0].message.content, chunks

## Additional Tools
Utility tools for notifications, logging, and user information handling.

In [17]:
def push(text):
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token": os.getenv("PUSHOVER_TOKEN"),
            "user": os.getenv("PUSHOVER_USER"),
            "message": text,
        }
    )

In [18]:
def record_user_details(email:str, name:str="Name not provided", notes:str="not provided")-> dict:
    """Store user contact details and notify via push"""
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [19]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [20]:
def record_unknown_question(question: str)-> dict:
    """Store unknown user question and notify via push"""
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [21]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

## Tool-Based RAG Function
Main callable function used by the assistant for answering questions with RAG.

In [22]:
def answer_with_rag(question: str, history: list[dict] = None) -> dict:
    answer, chunks = answer_question(question, history)

    return {
        "answer": answer,
        "context_snippets": [c.page_content[:300] for c in chunks],
        "sources": [c.metadata.get("source", "unknown") for c in chunks]
    }

In [23]:
answer_with_rag_json = {
    "name": "answer_with_rag",
    "description": "Answer user questions using RAG over Joanna Waligóra projects. Returns answer and retrieved context snippets and sources.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The user's question"
            },
            "history": {
                "type": "array",
                "description": "Chat history in OpenAI format (role/content dicts)",
                "items": {
                    "type": "object",
                    "properties": {
                        "role": {"type": "string"},
                        "content": {"type": "string"}
                    },
                    "required": ["role", "content"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [24]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json},
        {"type": "function", "function": answer_with_rag_json }]

In [25]:
def handle_tool_call(self, tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print(f"Tool called: {tool_name}", flush=True)

        tool = globals().get(tool_name)

        if tool is None:
            result = {"error": f"Tool {tool_name} not found"}
        else:
            try:
                result = tool(**arguments)
            except Exception as e:
                result = {"error": str(e)}

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

    return results

## LinkedIn Context Loading
LinkedIn PDF content is extracted and injected into the assistant system prompt.

In [ ]:
reader = PdfReader("linkedin.pdf")

linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

## Main Assistant Prompt
Defines the behavior and rules for the portfolio assistant.

In [27]:
main_system_prompt = f"""
You are Joanna Waligóra's portfolio assistant.

{linkedin}

RULES:
- LinkedIn = career truth
- RAG = GitHub project truth
- For project questions ALWAYS use RAG first
- Never invent projects or skills
- If information is missing → say "I don't know"

Use RAG for:
- projects
- tech stack
- models
- implementation details
- portfolio questions

Be concise and factual.
"""

## Chat Interface
Creates the Gradio interface for interacting with the assistant.

In [28]:
def chat(message, history):
    messages = [{"role": "system", "content": main_system_prompt}]

    for h in history:
        messages.append({
            "role": "assistant" if h["role"] == "assistant" else "user",
            "content": h["content"]
        })

    messages.append({
        "role": "user",
        "content": message
    })

    for _ in range(2):

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if msg.tool_calls:

            messages.append(msg)

            tool_results = handle_tool_calls(msg.tool_calls)

            messages.extend(tool_results)

        else:
            return msg.content or "No response generated."

    return "Tool loop limit reached."

In [29]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Baseline Evaluation

This section evaluates the initial retrieval and answer generation performance before optimization.

In [30]:
TEST_FILE = "tests.jsonl"

In [31]:
class TestQuestion(BaseModel):
    """A test question with expected keywords and reference answer."""

    question: str = Field(description="The question to ask the RAG system")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="Question category (e.g., atomic_fact, cross_context, comparative_reasoning)")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")


In [32]:
def load_tests() -> list[TestQuestion]:
    """Load test questions from JSONL file."""
    tests = []
    with open(TEST_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            tests.append(TestQuestion(**data))
    return tests

In [33]:
tests = load_tests()

In [34]:
example = tests[4]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

In Car_brand_classification_CNN, how many epochs is the model trained for before early stopping is applied?
atomic_fact
The model is trained for 100 epochs with early stopping applied during training.
['epochs', 'training', 'early stopping']


In [35]:
count = Counter([t.category for t in tests])
count

Counter({'atomic_fact': 101,
         'comparative_reasoning': 18,
         'local_reasoning': 11,
         'cross_context': 3})

## Retrieval Metrics
Implements metrics such as MRR, nDCG, and coverage.

In [36]:
def calculate_mrr(keyword, docs):
    keyword = keyword.lower()
    for i, doc in enumerate(docs, start=1):
        if keyword in doc.page_content.lower():
            return 1 / i
    return 0.0


def calculate_dcg(relevances, k):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances[:k]))


def calculate_ndcg(keyword, docs, k=10):
    keyword = keyword.lower()
    relevances = [1 if keyword in doc.page_content.lower() else 0 for doc in docs[:k]]
    
    dcg = calculate_dcg(relevances, k)
    idcg = calculate_dcg(sorted(relevances, reverse=True), k)
    
    return dcg / idcg if idcg > 0 else 0.0

In [37]:
def evaluate_retrieval_one(test):
    docs = fetch_context(test.question)

    mrrs = [calculate_mrr(k, docs) for k in test.keywords]
    ndcgs = [calculate_ndcg(k, docs) for k in test.keywords]

    return {
        "mrr": sum(mrrs)/len(mrrs),
        "ndcg": sum(ndcgs)/len(ndcgs),
        "coverage": sum(1 for x in mrrs if x > 0) / len(mrrs) * 100
    }

## Answer Evaluation
LLM-based evaluation of generated answers.

In [38]:
def evaluate_answer_one(test):
    answer, docs = answer_question(test.question)

    messages = [
        {
            "role": "system",
            "content": """
You are a STRICT evaluation system for RAG answers.

You MUST evaluate based on the REFERENCE ANSWER only.

IMPORTANT RULES:
- Do NOT use external knowledge.
- Do NOT guess facts.
- Only compare GENERATED ANSWER vs REFERENCE ANSWER.

You evaluate 3 independent dimensions:

1. accuracy:
- Is the final answer factually correct?
- 5 = exact correct final answer
- 1 = wrong answer

2. completeness:
- Does the answer include ALL key information from reference?
- 5 = fully covers reference answer
- 1 = missing most key info

3. relevance:
- Does the answer directly address the question?
- 5 = fully on-topic, no hallucinations
- 1 = off-topic or irrelevant

Return ONLY valid JSON in this exact format:
{
  "accuracy": int,
  "completeness": int,
  "relevance": int
}
"""
        },
        {
            "role": "user",
            "content": f"""
QUESTION:
{test.question}

REFERENCE ANSWER:
{test.reference_answer}

GENERATED ANSWER:
{answer}
"""
        }
    ]

    resp = completion(
        model="gpt-4.1-mini",
        messages=messages,
        response_format={"type": "json_object"}
    )

    try:
        scores = json.loads(resp.choices[0].message.content)

        return {
            "accuracy": scores.get("accuracy", 1),
            "completeness": scores.get("completeness", 1),
            "relevance": scores.get("relevance", 1),
        }

    except Exception:
        return {
            "accuracy": 1,
            "completeness": 1,
            "relevance": 1,
        }

## Full Evaluation Run
Runs retrieval and answer evaluation over the full test dataset.

In [39]:
# RETRIEVAL
mrr_total = 0
ndcg_total = 0
cov_total = 0

# ANSWERS
acc_total = 0
comp_total = 0
rel_total = 0

for test in tests:
    r = evaluate_retrieval_one(test)
    mrr_total += r["mrr"]
    ndcg_total += r["ndcg"]
    cov_total += r["coverage"]

    a = evaluate_answer_one(test)
    acc_total += a["accuracy"]
    comp_total += a["completeness"]
    rel_total += a["relevance"]

n = len(tests)

print("=== RETRIEVAL ===")
print("MRR:", round(mrr_total/n, 4))
print("nDCG:", round(ndcg_total/n, 4))
print("Coverage:", round(cov_total/n, 2), "%")

print("\n=== ANSWERS ===")
print("Accuracy:", round(acc_total/n, 2))
print("Completeness:", round(comp_total/n, 2))
print("Relevance:", round(rel_total/n, 2))

=== RETRIEVAL ===
MRR: 0.3609
nDCG: 0.4142
Coverage: 64.33 %

=== ANSWERS ===
Accuracy: 4.53
Completeness: 4.36
Relevance: 4.89


# Baseline Conclusions

This notebook validated the initial end-to-end implementation of the AI Portfolio Assistant.

The baseline system demonstrated that repository-based Retrieval-Augmented Generation can successfully answer questions about portfolio projects.

Main observations:

- retrieval quality strongly influenced answer quality,
- document chunking affected retrieved context,
- generated answers depended on retrieved information,
- additional optimization opportunities were identified.

This version served as the foundation for the next iteration of the project.

The following notebook (career_chat_2) focused on improving retrieval quality, reducing noisy context, and evaluating different optimization strategies.